# L2: Classification, Regression & Forecasting with TabPFN

In this lesson, you will make your first predictions with TabPFN. You will use `TabPFNClassifier` to assess heart-disease risk, `TabPFNRegressor` to predict house prices and inspect the uncertainty intervals behind every regression output, before applying TabPFN to time-series forecasting.

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">


<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download"</em>.</p>
</div>

## 1. Setup

Import the evaluation metrics, dataset loaders, and plotting helpers, then authenticate your `tabpfn_client` session to use `TabPFNClassifier` and `TabPFNRegressor`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    mean_absolute_error,
    precision_score,
    r2_score,
    recall_score,
)

from helper import (
    configure_lesson,
    init_tabpfn,
    load_electricity,
    load_heart_disease_full,
    load_housing,
    plot_actual_vs_predicted,
    plot_electricity,
    plot_forecast_comparison,
    plot_predicted_distribution,
    plot_quantile_band,
)

configure_lesson(plots=True)

from tabpfn_client import TabPFNClassifier, TabPFNRegressor
init_tabpfn()

## 2. Classification: is this patient at risk?

Let's start with binary classification on clinical data:

- **The dataset:** 920 patients pooled from four hospital studies (Cleveland, Hungary, Switzerland, and Long Beach), split into 700 patients for training context and 220 held out for testing.
- **The target:** Binary (`1` = coronary heart disease, `0` = no disease).
- **Missing values:** Roughly two-thirds of the records have missing measurements (`NaN`). TabPFN handles missing values natively—no imputation needed.

In [ ]:
X_train, X_test, y_train, y_test = load_heart_disease_full(
    train_size=700, test_size=220,
)

X_train.head(3)

In [ ]:
y_train.head(3)

The interface looks just like scikit-learn's, but the mechanics under the hood are different:

- **Pretrained foundation model:** `fit` does **no** parameter training—it simply prepares and uploads your training table so the model can use it as context.
- **In-context learning:** All model compute happens at `predict` time. The transformer reads the training rows and new test rows together to predict in a single forward pass.
- **Repeated inference:** Because the work happens at `predict` time, repeated requests re-encode the same training context. Lesson 4 shows how a KV-cache removes that extra cost.

No hyperparameter tuning or manual feature encoding is needed: TabPFN automatically detects categorical columns and works out of the box.

In [ ]:
classifier = TabPFNClassifier()
classifier.fit(X_train, y_train)

### Predict one patient first

Before scoring the whole test set, let's predict a single patient. `predict` returns the binary class label (`0` or `1`), while `predict_proba` returns the calibrated probability behind that decision—which is the number a clinician would actually use to assess risk.

In [ ]:
patient = X_test.iloc[[0]]
display(patient)

In [ ]:
prediction = classifier.predict(patient)[0]


print(f"predicted: {prediction:.0f}   "
      f"actual: {y_test.iloc[0]:.0f}")

In [ ]:
proba = classifier.predict_proba(patient)[0]
print(f"P(healthy: label=0) = {proba[0]:.2f}   "
      f"P(disease: label=1) = {proba[1]:.2f}")

### Evaluate on all 220 held-out patients

One patient tells a story; 220 patients tell us how well the model generalizes. Instead of compressing performance into a single accuracy score, let's inspect the **confusion matrix**: each row is the patient's true status, and each column is the model's prediction. The diagonal cells show correct predictions, while the off-diagonal cells show the two types of errors—which carry very different costs in a clinical setting.

In [ ]:
predictions = classifier.predict(X_test)

ConfusionMatrixDisplay.from_predictions(
    y_test,
    predictions,
    display_labels=["no disease", "disease"],
    cmap="Blues",
    colorbar=False,
);


Two metrics summarize those two types of errors:

- **Precision:** Of the patients the model flags as having heart disease, what fraction actually have it? Low precision means more false alarms and unnecessary follow-up tests.
- **Recall:** Of the patients who truly have heart disease, what fraction does the model catch? Low recall means missed diagnoses—the costlier mistake in a clinic.

In [ ]:
precision = precision_score(y_test, predictions)
recall = recall_score(y_test, predictions)

print(f"precision: {precision:.2f}")
print(f"recall:    {recall:.2f}")

## 3. Regression: house price prediction

Now let's switch to a continuous target: predicting the sale price of houses in Ames, Iowa from a handful of property features. On your end, the only code change is swapping `TabPFNClassifier` for `TabPFNRegressor`.

- **The dataset:** The [Ames Housing dataset](https://www.openml.org/d/42165) (De Cock, 2011)—1,460 home sales from 2006–2010 using 9 of the original 80 features (1,300 for training context, 160 for testing).
- **Mixed data types:** Notice that the `Neighborhood` column contains raw text strings. You don't need one-hot encoding or preprocessing—TabPFN handles text and numeric columns together natively.

In [ ]:
X_train_h, X_test_h, y_train_h, y_test_h = load_housing(
    train_size=1300,
    test_size=160,
)
X_train_h.head(3)

In [ ]:
y_train_h.head(3)

In [ ]:
reg = TabPFNRegressor()
reg.fit(X_train_h, y_train_h)

### Predict the price of one house first

A traditional regression model answers *"What is this house worth?"* with a single number. TabPFN answers a richer question: *"How likely is each possible price?"* Internally, it divides the price range into thousands of bins and predicts a probability for each, producing a full probability distribution over prices.

Every output from `predict` is a summary of that distribution:

- **Default (`mean`):** The expected sale price.
- **Quantiles (`output_type="quantiles"`):** For example, the **10th** and **90th percentiles** give you the range where the model places 80% of its probability mass, with the **50th percentile** as the median.

In [ ]:
house_ind = 5
house = X_test_h.iloc[[house_ind]]
display(house)

In [ ]:
mean_price = reg.predict(house)[0]
print(f"mean:            ${mean_price:,.0f}")
print(f"actual price:    ${y_test_h.iloc[house_ind]:,.0f}")


q10, q50, q90 = reg.predict(
    house,
    output_type="quantiles",
    quantiles=[0.1, 0.5, 0.9],
)

print(f"10th percentile: ${q10[0]:,.0f}")
print(f"median:          ${q50[0]:,.0f}")
print(f"90th percentile: ${q90[0]:,.0f}")


Think of this output as the model saying: *"My best single estimate is the mean, and I estimate an 80% chance the true sale price falls between the 10th and 90th percentiles."* Two houses can have the exact same predicted mean but very different uncertainty bands: a narrow band means the model is confident, while a wide band signals higher uncertainty. A single point estimate hides that distinction.

### Evaluate on the whole test set

For a continuous target, we evaluate how close the predictions are to the actual sale prices:

- **MAE (Mean Absolute Error):** How far off the predictions are on average, measured in the target's own units (dollars).
- **R² (Coefficient of Determination):** The share of house-to-house price variation explained by the model (`1.0` is a perfect fit; `0.0` is no better than guessing the average price every time).

In [ ]:
point = reg.predict(X_test_h)

comparison = pd.DataFrame(
    {
        "predicted ($1000s)": (point[:6] / 1000).round(0),
        "actual ($1000s)": (y_test_h.values[:6] / 1000),
    }
)
print(comparison)

* Evaluating errors

Let's compute the overall MAE and R² across all 160 test houses:

In [ ]:
mae = mean_absolute_error(y_test_h, point)
print(f"MAE: ${mae:,.0f}")
print(f"R2:  {r2_score(y_test_h, point):.2f}")

We can also visualize this comparison as a scatter plot: each dot is one test house, with the actual price on the x-axis and the predicted price on the y-axis. A perfect model would place every dot right on the dashed diagonal line; the vertical distance from the line is that house's prediction error.

In [ ]:
plot_actual_vs_predicted(
    y_test_h / 1000,
    point / 1000,
    unit="sale price ($1000s)",
)

### Can you trust the uncertainty band?

An "80% prediction interval" is only meaningful if the model is well **calibrated**: across many test houses, the true sale price should fall inside the 10th–90th percentile band about 80% of the time. Let's check the actual empirical coverage on our 160 held-out houses and plot their uncertainty bands sorted by predicted median price.

In [ ]:
low, median, high = reg.predict(
    X_test_h,
    output_type="quantiles",
    quantiles=[0.1, 0.5, 0.9]
)

actual = y_test_h.values
inside = (actual >= low) & (actual <= high)
print(f"the 10-90 band covers {inside.mean():.0%} "
      f"of {len(actual)} test houses")

plot_quantile_band(
    actual / 1000,
    low / 1000,
    median / 1000,
    high / 1000,
    "Sale price ($1000s)",
    unit="test houses",
)

## 4. Time-series forecasting

TabPFN has no built-in notion of time—it treats rows as an unordered table. However, forecasting is just regression where future rows don't have a target value yet! The key is **how we represent time as features**, because the model can only learn patterns from the columns we give it.

- **The dataset:** Two weeks of hourly German day-ahead electricity prices (`price_eur`), which follow a strong daily cycle, a weekly rhythm, and occasional price spikes. We use the first 336 hours as history and hold out the final 48 hours to forecast.

In [ ]:
history, future = load_electricity(horizon=48)

print(f"history:  {len(history)} hours")
print(f"held out: {len(future)} hours")

plot_electricity(history)

#### Turn the calendar into features

* Baseline: forecasting from the raw timestamp

First, let's inspect the table and see what happens if we pass only the raw `hour` timestamp column to `TabPFNRegressor` without extracting any cyclical calendar features.

In [ ]:
history.head(3)

#### Prediction

In [ ]:
history[["hour","price_eur"]].head(3)

In [ ]:
reg_ts = TabPFNRegressor()

reg_ts.fit(history[["hour"]], history["price_eur"])
forecast_price = reg_ts.predict(future[["hour"]])

In [ ]:
low_price, _, high_price = reg_ts.predict(future[["hour"]],
                                          output_type="quantiles", 
                                          quantiles=[0.1, 0.5, 0.9])

plot_forecast_comparison(
    history, future,
    {"forecast": forecast_price}, time_col="hour",
    target_col="price_eur", ylabel="price",
    title="48 hours of prices, forecast from the clock",
    band=(low_price, high_price),
)

### Turn the clock into features

With only a raw timestamp, every future hour looks like a brand-new number outside the training range, so the forecast flattens out. To help the model see the repeating patterns, let's extract two simple calendar features from the timestamp:

- `hour_of_day` (`0`–`23`) places each row on the daily cycle.
- `day_of_week` (`0`–`6`) separates weekdays from the weekend.

In [ ]:
def add_calendar_features(df):
    out = df.copy()
    out["hour_of_day"] = out["hour"].dt.hour
    out["day_of_week"] = out["hour"].dt.dayofweek
    return out


FEATURES = ["hour_of_day", "day_of_week"]

history_cal = add_calendar_features(history)
future_cal = add_calendar_features(future)

reg_ts = TabPFNRegressor()
reg_ts.fit(history_cal[FEATURES], history_cal["price_eur"])

forecast_price = reg_ts.predict(future_cal[FEATURES])

And because every TabPFN regression outputs a full distribution, our time-series forecast automatically comes with a 10th–90th percentile uncertainty band at no extra compute cost:

In [ ]:
low_price, _, high_price = reg_ts.predict(future_cal[FEATURES],
                                          output_type="quantiles",
                                          quantiles=[0.1, 0.5, 0.9])

plot_forecast_comparison(
    history, future,
    {"calendar features": forecast_price}, time_col="hour",
    target_col="price_eur", ylabel="price",
    title="48 hours of prices, forecast from the clock",
    band=(low_price, high_price),
)

### Exercise: forecast with more than the clock

Electricity prices depend heavily on power demand and renewable energy supply. Because grid operators publish day-ahead forecasts for both, those values (`load_forecast` and `renewables_forecast`) are already available in our table for the future hours we want to predict!

**Your turn:**
1. Add `"load_forecast"` and `"renewables_forecast"` to `FEATURES` and fit a new `TabPFNRegressor`.
2. Compare its MAE on `future["price_eur"]` against the calendar-only forecast above.
3. Plot both forecasts together using `plot_forecast_comparison`.

In [ ]:
# YOUR CODE HERE
#
# Hint: MARKET_FEATURES = FEATURES + ["load_forecast",
# "renewables_forecast"], fit a second TabPFNRegressor on
# history_cal[MARKET_FEATURES], predict future_cal, compare
# mean_absolute_error on future["price_eur"], then pass both
# forecasts to plot_forecast_comparison in one dict.

Notice that the model architecture never changed—only its context did. With a tabular foundation model, **feature engineering is context engineering**: you decide what information the model gets to read before it predicts. Calendar features give it the clock; exogenous forecasts give it the market conditions. (The `tabpfn-time-series` package automates this exact workflow.)

## Recap

In this lesson, you used TabPFN across three core tabular tasks:

- **Classification & Regression with one API:** Use `TabPFNClassifier` and `TabPFNRegressor` with a familiar scikit-learn interface—no hyperparameter tuning, missing-value imputation, or manual categorical encoding required.
- **In-context learning:** `fit` uploads the training table as context, and all model computation happens at `predict` time (Lesson 4 shows how to cache that context for fast real-time serving).
- **Built-in uncertainty:** Every regression output is a full probability distribution, giving you calibrated quantile intervals and uncertainty bands for free.
- **Forecasting via context engineering:** Giving the model explicit calendar features and day-ahead exogenous variables turns time-series forecasting into a standard tabular regression problem.